# Task 1.1



In [36]:
import csv
import json

with open("/content/appointments.csv", "r", newline="")as file :
  reader = csv.DictReader(file)

  rows = list(reader)

  print('header:', reader.fieldnames)
  print('Total Rows :', len(rows))

header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total Rows : 1020


In [37]:
def parse_row(row):
    """
    Purpose: Converts numeric columns in one appointment row.

    Parameter:
        row (dict): One appointment record from the CSV.

    Returns:
        dict: The same row with converted numeric values.
    """
    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])

    return row

In [38]:
accepted = []
rejected = []

for row in rows:

    try:
        converted = parse_row(row)
        accepted.append(converted)

    except ValueError:
        rejected.append(row)

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))
print("Total:", len(accepted) + len(rejected))

Accepted: 988
Rejected: 32
Total: 1020


In [39]:
with open("rejected_rows.json", "w") as file:

    json.dump(rejected, file, indent=2)

print("Rejected records saved successfully:", len(rejected))

Rejected records saved successfully: 32


# Task 1.2



In [40]:
import pandas as pd

appointments = pd.read_csv("/content/appointments.csv")

print("Shape:", appointments.shape)


print("\nDataset Information:")
appointments.info()

print("\nMissing Values:")
print(appointments.isnull().sum())

print("\nExact Duplicates:")
print(appointments.duplicated().sum())

print("\nInitial Row Count:", len(appointments))

Shape: (1020, 14)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing Values:
appointment_id      0
patient_id   

In [41]:
appointments = appointments.drop_duplicates()

print("Rows after dropping duplicates:", len(appointments))


appointments["booking_channel"] = appointments["booking_channel"].str.strip().str.title()

print("Rows after cleaning booking_channel:", len(appointments))
print("Unique booking channels:", appointments["booking_channel"].unique())

Rows after dropping duplicates: 1000
Rows after cleaning booking_channel: 1000
Unique booking channels: ['Phone' 'Walk-In' 'App']


In [42]:
appointments["appt_date"] = pd.to_datetime(appointments["appt_date"], errors="coerce")

appointments["booked_on"] = pd.to_datetime(appointments["booked_on"])

print("Rows after date conversion:", len(appointments))

Rows after date conversion: 1000


In [43]:
appointments = appointments.dropna(subset=["appt_date", "status"])

print("Rows after dropping missing dates/status:", len(appointments))

Rows after dropping missing dates/status: 990


# Task 1.3


In [44]:
import numpy as np

invalid_age = (appointments["age"] < 1) | (appointments["age"] > 100)

print("Invalid ages:", invalid_age.sum())

appointments.loc[invalid_age, "age"] = np.nan

print("Missing ages:", appointments["age"].isnull().sum())

Invalid ages: 5
Missing ages: 17


In [45]:
age_median = appointments["age"].median()
distance_median = appointments["distance_km"].median()

print("Median Age:", age_median)
print("Median Distance:", distance_median)


appointments["age"] = appointments["age"].fillna(age_median).round().astype(int)


appointments["distance_km"] = appointments["distance_km"].fillna(distance_median)


print("Missing Age:", appointments["age"].isnull().sum())
print("Missing Distance:", appointments["distance_km"].isnull().sum())

Median Age: 48.0
Median Distance: 8.25
Missing Age: 0
Missing Distance: 0


In [46]:
distance_q1 = appointments["distance_km"].quantile(0.25)
distance_q3 = appointments["distance_km"].quantile(0.75)
distance_iqr = distance_q3 - distance_q1


distance_lower = distance_q1 - 1.5 * distance_iqr
distance_upper = distance_q3 + 1.5 * distance_iqr


fee_q1 = appointments["fee_inr"].quantile(0.25)
fee_q3 = appointments["fee_inr"].quantile(0.75)
fee_iqr = fee_q3 - fee_q1


fee_lower = fee_q1 - 1.5 * fee_iqr
fee_upper = fee_q3 + 1.5 * fee_iqr


print("Distance:", distance_q1, distance_q3, distance_iqr)
print("Distance limits:", distance_lower, distance_upper)

print("Fee:", fee_q1, fee_q3, fee_iqr)
print("Fee limits:", fee_lower, fee_upper)

Distance: 4.625 13.0 8.375
Distance limits: -7.9375 25.5625
Fee: 500.0 787.5 287.5
Fee limits: 68.75 1218.75


In [47]:
np.clip?

In [48]:
distance_capped = (
    (appointments["distance_km"] < distance_lower) |
    (appointments["distance_km"] > distance_upper)
).sum()


fee_capped = (
    (appointments["fee_inr"] < fee_lower) |
    (appointments["fee_inr"] > fee_upper)
).sum()


appointments["distance_km"] = np.clip(
    appointments["distance_km"], distance_lower, distance_upper
)

appointments["fee_inr"] = np.clip(
    appointments["fee_inr"], fee_lower, fee_upper
)


print("Distance values capped:", distance_capped)
print("Fee values capped:", fee_capped)

Distance values capped: 26
Fee values capped: 3


In [49]:
appointments["distance_km"] = appointments["distance_km"].round(1)


appointments["fee_inr"] = appointments["fee_inr"].round().astype(int)


print("Largest fee:", appointments["fee_inr"].max())

print("Final rows:", len(appointments))

print(appointments['distance_km'].dtypes, appointments['fee_inr'].dtypes)

Largest fee: 1219
Final rows: 990
float64 int64


# Task 1.4


In [50]:
appointments["lead_days"] = (
    appointments["appt_date"] - appointments["booked_on"]
).dt.days


print(appointments["lead_days"].head())


print("Minimum lead days:", appointments["lead_days"].min())
print("Maximum lead days:", appointments["lead_days"].max())

0     1
1    10
2     6
4     4
5    40
Name: lead_days, dtype: int64
Minimum lead days: 1
Maximum lead days: 45


In [51]:
appointments["no_show"] = (appointments["status"] == "No-Show").astype(int)


print(appointments[["status", "no_show"]].head())

print(appointments["no_show"].value_counts())



     status  no_show
0   No-Show        1
1  Attended        0
2  Attended        0
4  Attended        0
5   No-Show        1
no_show
0    772
1    218
Name: count, dtype: int64


In [52]:
appointments["no_show"] = (appointments["status"] == "No-Show").astype(int)


print(appointments[["status", "no_show"]].head())

print(appointments["no_show"].value_counts())

     status  no_show
0   No-Show        1
1  Attended        0
2  Attended        0
4  Attended        0
5   No-Show        1
no_show
0    772
1    218
Name: count, dtype: int64


In [53]:
X = appointments[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()


print(X)

print("Array shape:", X.shape)

[[ 60.   18.  850.    1. ]
 [ 39.   12.5 350.   10. ]
 [ 63.    4.1 850.    6. ]
 ...
 [ 58.    3.2 850.   11. ]
 [ 59.   25.6 300.   13. ]
 [ 38.   10.8 550.   45. ]]
Array shape: (990, 4)


In [54]:
column_means = np.mean(X, axis=0)

print("Column means:", column_means.round(2))

Column means: [ 48.17   9.5  635.71  11.82]


In [55]:
mask = appointments["lead_days"] > 14

count = mask.sum()

share = mask.mean()

print("Appointments above 14 days:", count)
print("Share:", round(share * 100, 1), "%")

Appointments above 14 days: 296
Share: 29.9 %


In [56]:
x = appointments["distance_km"].to_numpy()

z = (x - np.mean(x)) / np.std(x)

print("First five z-scores:", z[:5].round(2))

First five z-scores: [ 1.38  0.49 -0.88 -0.32  1.54]


In [57]:
outlier_mask = np.abs(z) > 2

outlier_count = outlier_mask.sum()


print("Distance z-score outliers:", outlier_count)

Distance z-score outliers: 53


# Task 1.5



In [58]:
clinics = pd.read_csv("/content/clinics.csv")

print(clinics.head())

print("\nShape:", clinics.shape)

print("\nColumns:", clinics.columns.tolist())



  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873

Shape: (8, 6)

Columns: ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']


In [ ]:
import requests

latitude = clinics.loc[0, "latitude"]
longitude = clinics.loc[0, "longitude"]

response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1
    },
    timeout=10
)

print("Status code:", response.status_code)

In [ ]:
weather_data = response.json()

temperature_list = weather_data["hourly"]["temperature_2m"]

temp_c = temperature_list[0]

print("Clinic:", clinics.loc[0, "clinic_name"])
print("Temperature:", temp_c, "°C")

In [ ]:
import requests

temperatures = []

for index, clinic in clinics.iterrows():

    try:
        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": clinic["latitude"],
                "longitude": clinic["longitude"],
                "hourly": "temperature_2m",
                "forecast_days": 1
            },
            timeout=10
        )

        if response.status_code != 200:
            raise ValueError(f"HTTP status {response.status_code}")

        weather_data = response.json()

        if "hourly" not in weather_data or not weather_data["hourly"].get("temperature_2m"):
            raise ValueError("Hourly temperature data missing")

        temp_c = weather_data["hourly"]["temperature_2m"][0]

    except requests.exceptions.RequestException as error:
        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={
                    "latitude": clinic["latitude"],
                    "longitude": clinic["longitude"],
                    "hourly": "temperature_2m",
                    "forecast_days": 1
                },
                timeout=10
            )

            if response.status_code != 200:
                raise ValueError(f"HTTP status {response.status_code}")

            weather_data = response.json()

            if "hourly" not in weather_data or not weather_data["hourly"].get("temperature_2m"):
                raise ValueError("Hourly temperature data missing")

            temp_c = weather_data["hourly"]["temperature_2m"][0]

        except (requests.exceptions.RequestException, ValueError, KeyError) as retry_error:
            temp_c = None
            print(f"Failed for {clinic['clinic_name']}: {retry_error}")

    except (ValueError, KeyError) as error:
        temp_c = None
        print(f"Failed for {clinic['clinic_name']}: {error}")

    temperatures.append(temp_c)

    print(clinic["clinic_name"], ":", temp_c)

Kothrud Family Clinic : 15.8
Hadapsar Care Centre : 16.9
Sitabuldi Health Point : 18.4
Malviya Nagar Clinic : 15.9
Vaishali Nagar Clinic : 15.9
Edappally Medical Centre : 25.7
Vijay Nagar Clinic : 14.3
Gomti Nagar Care Point : 16.1


In [ ]:
clinics["temp_c"] = temperatures

print(clinics)

clinics.to_csv("clinics_enriched.csv", index=False)

print("\nShape:", clinics.shape)

print("\nTemperature data type:", clinics["temp_c"].dtype)
print("Missing temperatures:", clinics["temp_c"].isnull().sum())

# Task 1.6


In [62]:
print("Unique appointment IDs:", appointments["appointment_id"].nunique())

print("Duplicate appointment IDs:", appointments["appointment_id"].duplicated().sum())

print("\nMissing values in each column:")
print(appointments.isnull().sum())

print("\nTotal missing values:", appointments.isnull().sum().sum())

print("\nTotal appointment records:", len(appointments))

Unique appointment IDs: 990
Duplicate appointment IDs: 0

Missing values in each column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

Total missing values: 0

Total appointment records: 990


In [63]:
appointments = appointments.sort_values(by="appointment_id")

appointments["booked_on"] = appointments["booked_on"].dt.strftime("%Y-%m-%d")

appointments["appt_date"] = appointments["appt_date"].dt.strftime("%Y-%m-%d")

required_columns = [
    "appointment_id", "patient_id", "clinic_id", "booked_on",
    "appt_date", "age", "gender", "distance_km", "fee_inr",
    "booking_channel", "visit_type", "reminder_sent",
    "prior_no_shows", "status", "lead_days", "no_show"
]

appointments_clean = appointments[required_columns].copy()


appointments_clean.to_csv("appointments_clean.csv", index=False)

print("Final shape:", appointments_clean.shape)
print("Columns:", appointments_clean.columns.tolist())

Final shape: (990, 16)
Columns: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']


In [64]:
appointments_clean.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
4,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
5,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [65]:
appointments_clean = pd.read_csv('/content/appointments_clean.csv')
clinics_enriched = pd.read_csv('/content/clinics_enriched.csv')

print("Appointments:", appointments_clean.shape)
print("Clinics:", clinics_enriched.shape)

Appointments: (990, 16)
Clinics: (8, 7)


In [66]:
print("Clinic columns:", clinics_enriched.columns.tolist())
print("Appointment columns:", appointments_clean.columns.tolist())

Clinic columns: ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude', 'temp_c']
Appointment columns: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']


In [67]:
def format_value(value):
    """
    Convert a Python value into SQL-compatible format.

    Parameters:
        value: The input value (string, number, or missing value).

    Returns:
        str: 'NULL' for missing values,
             quoted text for strings,
             unquoted text for numbers.
    """
    if pd.isna(value):
        return "NULL"

    elif isinstance(value, str):
        return f"'{value}'"

    else:
        return str(value)

In [68]:
with open("/content/load_data.sql", "w") as file:

    for _, row in clinics_enriched.iterrows():

        formatted_values = []

        for value in row:


            formatted_value = format_value(value)

            formatted_values.append(formatted_value)

        values = ", ".join(formatted_values)

        sql = f"INSERT INTO clinics VALUES ({values});\n"

        file.write(sql)

In [69]:
with open("/content/load_data.sql", "a") as file:

    for _, row in appointments_clean.iterrows():

        formatted_values = []

        for value in row:

            formatted_value = format_value(value)

            formatted_values.append(formatted_value)

        values = ", ".join(formatted_values)

        sql = f"INSERT INTO appointments VALUES ({values});\n"


        file.write(sql)

In [70]:
with open("/content/load_data.sql", "r") as file:
    sql_content = file.readlines()


print("Total INSERT statements:", len(sql_content))

Total INSERT statements: 998


In [71]:
clinic_count = sum(
    line.startswith("INSERT INTO clinics")
    for line in sql_content
)

appointment_count = sum(
    line.startswith("INSERT INTO appointments")
    for line in sql_content
)


print("Clinic statements:", clinic_count)
print("Appointment statements:", appointment_count)
print("Total statements:", len(sql_content))

print("First statement:", sql_content[0])
print("Last statement:", sql_content[-1])

Clinic statements: 8
Appointment statements: 990
Total statements: 998
First statement: INSERT INTO clinics VALUES ('C01', 'Kothrud Family Clinic', 'Pune', 'Metro', 18.5074, 73.8077, 15.8);

Last statement: INSERT INTO appointments VALUES ('A1000', 'P085', 'C08', '2024-12-03', '2025-01-17', 38, 'M', 10.8, 550, 'App', 'Follow-up', 'Yes', 2, 'No-Show', 45, 1);



# Task 2.2

In [72]:
import pandas as pd

appointments_clean = pd.read_csv("/content/appointments_clean.csv")

clinics_enriched = pd.read_csv("/content/clinics_enriched.csv")

In [73]:
clinic_groups = appointments_clean.groupby("clinic_id")

appointment_counts = clinic_groups["appointment_id"].count()

print(appointment_counts)





clinic_id
C01    160
C02    143
C03     82
C04    142
C05    130
C06    142
C07    101
C08     90
Name: appointment_id, dtype: int64


In [74]:
no_show_counts = clinic_groups["no_show"].sum()
print(no_show_counts)

clinic_id
C01    36
C02    39
C03    13
C04    38
C05    18
C06    23
C07    21
C08    30
Name: no_show, dtype: int64


In [75]:
clinic_summary = appointments_clean.groupby("clinic_id").agg(
    appointments=("appointment_id", "count"),
    no_shows=("no_show", "sum")
)

clinic_summary["no_show_pct"] = (
    100 * clinic_summary["no_shows"] / clinic_summary["appointments"]
).round(1)

print(clinic_summary.sort_values(by="no_show_pct", ascending=False))

           appointments  no_shows  no_show_pct
clinic_id                                     
C08                  90        30         33.3
C02                 143        39         27.3
C04                 142        38         26.8
C01                 160        36         22.5
C07                 101        21         20.8
C06                 142        23         16.2
C03                  82        13         15.9
C05                 130        18         13.8


# Task 3.1


In [ ]:
import pandas as pd

df = pd.read_csv("/content/appointments_clean.csv")

df.shape

(990, 16)

In [ ]:
df.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [ ]:
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()


X["reminder_sent"] = X["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)


y = df["no_show"]


print("X shape:", X.shape)
print("y shape:", y.shape)


print("\nFeature columns:")
print(X.columns.tolist())

X shape: (990, 11)
y shape: (990,)

Feature columns:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (792, 11)
Testing features: (198, 11)
Training target: (792,)
Testing target: (198,)


In [ ]:
from sklearn.metrics import accuracy_score

baseline_predictions = [0] * len(y_test)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)

test_no_shows = (y_test == 1).sum()

print("Test set size:", len(y_test))
print("Actual no-shows in test set:", test_no_shows)
print("Baseline accuracy:", round(baseline_accuracy, 4))

Test set size: 198
Actual no-shows in test set: 49
Baseline accuracy: 0.7525


# Task 3.2

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)

X_test_s = scaler.transform(X_test)

print("Scaled training shape:", X_train_s.shape)
print("Scaled testing shape:", X_test_s.shape)

Scaled training shape: (792, 11)
Scaled testing shape: (198, 11)


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)

print("Predictions:", y_pred[:10])

print("Actual:", y_test.iloc[:10].values)

Predictions: [0 0 1 0 0 0 0 0 0 0]
Actual: [0 0 0 0 0 1 0 1 0 0]


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

accuracy = accuracy_score(y_test, y_pred)

cm = confusion_matrix(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

y_prob = model.predict_proba(X_test_s)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)

print("Accuracy:", round(accuracy, 3))
print("\nConfusion Matrix:\n", cm)
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1-score:", round(f1, 3))
print("ROC-AUC:", round(roc_auc, 3))

Accuracy: 0.788

Confusion Matrix:
 [[139  10]
 [ 32  17]]
Precision: 0.63
Recall: 0.347
F1-score: 0.447
ROC-AUC: 0.789


In [ ]:
import numpy as np

feature_names = X.columns

coefficients = model.coef_[0]

for feature, coef in zip(feature_names, coefficients):
    print(feature, ":", round(coef, 3))

reminder_index = list(feature_names).index("reminder_sent")

reminder_coef = coefficients[reminder_index]

scaled_odds_ratio = np.exp(reminder_coef)

actual_odds_ratio = np.exp(
    reminder_coef / scaler.scale_[reminder_index]
)

print("\nReminder coefficient:", round(reminder_coef, 3))
print("Odds ratio (scaled):", round(scaled_odds_ratio, 3))
print("Odds ratio (No to Yes):", round(actual_odds_ratio, 3))

age : -0.254
distance_km : 0.253
fee_inr : 0.453
lead_days : 0.614
prior_no_shows : 0.65
reminder_sent : -0.581
booking_channel_App : -0.047
booking_channel_Phone : 0.216
booking_channel_Walk-In : -0.186
visit_type_Consultation : 0.016
visit_type_Follow-up : -0.016

Reminder coefficient: -0.581
Odds ratio (scaled): 0.56
Odds ratio (No to Yes): 0.309


The reminder_sent feature has the negative coefficient at -0.581. The exponential of this coefficient is 0.560, which shows how the odds change for a one-standard-deviation increase in the scaled feature. When we divide the coefficient by its deviation, which is about 0.49, we get an actual odds ratio of 0.309. This means that, in this model, appointments with reminders are linked to 69% lower odds of a no-show. This relationship is an association. It does not prove that sending reminders cause no-shows, because the data come from an observational study.

# Task 3.3


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_full = DecisionTreeClassifier(random_state=42)

dt_full.fit(X_train, y_train)

train_pred_full = dt_full.predict(X_train)

test_pred_full = dt_full.predict(X_test)

train_acc_full = accuracy_score(y_train, train_pred_full)

test_acc_full = accuracy_score(y_test, test_pred_full)

print("Full-depth Tree")
print("Training accuracy:", round(train_acc_full, 3))
print("Test accuracy:", round(test_acc_full, 3))
print("Tree depth:", dt_full.get_depth())

Full-depth Tree
Training accuracy: 1.0
Test accuracy: 0.672
Tree depth: 15


In [ ]:
dt_depth3 = DecisionTreeClassifier(    max_depth=3,random_state=42)

dt_depth3.fit(X_train, y_train)

train_pred_depth3 = dt_depth3.predict(X_train)

test_pred_depth3 = dt_depth3.predict(X_test)

train_acc_depth3 = accuracy_score(y_train, train_pred_depth3)

test_acc_depth3 = accuracy_score(y_test, test_pred_depth3)

print("Depth-3 Tree")
print("Training accuracy:", round(train_acc_depth3, 3))
print("Test accuracy:", round(test_acc_depth3, 3))
print("Tree depth:", dt_depth3.get_depth())

Depth-3 Tree
Training accuracy: 0.822
Test accuracy: 0.778
Tree depth: 3


The depth Decision Tree got 100% accuracy on the training data but only about 67% on the test data. That is a sign of overfitting. It even came in below the 75.25% baseline accuracy. The depth-3 Decision Tree got around 82% training accuracy and 78% test accuracy. I think that shows generalisation, because keeping the tree simple cut down on the overfitting.

# Task 3.4


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100,random_state=42)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_train_acc = accuracy_score(y_train, rf.predict(X_train))

rf_test_acc = accuracy_score(y_test, rf_pred)

rf_recall = recall_score(y_test, rf_pred)

rf_prob = rf.predict_proba(X_test)[:, 1]

rf_auc = roc_auc_score(y_test, rf_prob)

print("Training accuracy:", round(rf_train_acc, 3))
print("Test accuracy:", round(rf_test_acc, 3))
print("Recall:", round(rf_recall, 3))
print("ROC-AUC:", round(rf_auc, 3))

Training accuracy: 1.0
Test accuracy: 0.747
Recall: 0.306
ROC-AUC: 0.706


In [ ]:
importances = rf.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importances
})

feature_importance = feature_importance.sort_values(  by="Importance",ascending=False)

print(round(feature_importance.head(3),2))

       Feature  Importance
3    lead_days        0.22
0          age        0.20
1  distance_km        0.19


In [ ]:
# Create a comparison table for the three models
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest"
    ],

    "Accuracy": [
        accuracy,
        test_acc_depth3,
        rf_test_acc
    ],

    "Recall": [
        recall,
        recall_score(y_test, test_pred_depth3),
        rf_recall
    ],

    "ROC-AUC": [
        roc_auc,
        roc_auc_score(
            y_test,
            dt_depth3.predict_proba(X_test)[:, 1]
        ),
        rf_auc
    ]
})

print(model_comparison.round(3))

                     Model  Accuracy  Recall  ROC-AUC
0      Logistic Regression     0.788   0.347    0.789
1  Decision Tree (Depth 3)     0.778   0.224    0.700
2            Random Forest     0.747   0.306    0.706


# Task 3.5

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

rf_grid = GridSearchCV(
    rf_model,
    param_grid,
    cv=5,
    scoring="roc_auc"
)

rf_grid.fit(X_train, y_train)

print("Best Parameters:", rf_grid.best_params_)
print("Best CV ROC-AUC:", rf_grid.best_score_)

Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC: 0.7673242251739405


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

lr_auc = cross_val_score(
    lr_pipe, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

dt_auc = cross_val_score(
    dt_depth3, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

rf_auc = cross_val_score(
    rf, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

tuned_rf_auc = cross_val_score(
    rf_grid.best_estimator_, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

# Model comparison table

cv_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest",
        "Tuned Random Forest"
    ],
    "CV ROC-AUC": [
        lr_auc,
        dt_auc,
        rf_auc,
        tuned_rf_auc
    ]
})

print(cv_comparison.round(2))

                     Model  CV ROC-AUC
0      Logistic Regression        0.77
1  Decision Tree (Depth 3)        0.68
2            Random Forest        0.75
3      Tuned Random Forest        0.77


Logistic Regression got the cross-validation ROC-AUC of 0.77. It performed better than the Decision Tree, Random Forest and Tuned Random Forest. Because of this, Logistic Regression was chosen as the candidate model. It will now be used for threshold tuning and further evaluation.

In [ ]:
from sklearn.model_selection import cross_val_predict

oof = cross_val_predict(
    lr_pipe,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

print("Total probabilities:", len(oof))
print("First 10 probabilities:", oof[:10])

Total probabilities: 792
First 10 probabilities: [0.44277273 0.11344644 0.34663886 0.12072607 0.17356387 0.24988488
 0.12508639 0.4103534  0.16669732 0.08633667]


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

pred_50 = (oof >= 0.50).astype(int)

precision_50 = precision_score(y_train, pred_50)
recall_50 = recall_score(y_train, pred_50)
f1_50 = f1_score(y_train, pred_50)

flagged_50 = pred_50.sum()

print("Threshold: 0.50")
print("Precision:", round(precision_50, 3))
print("Recall:", round(recall_50, 3))
print("F1-score:", round(f1_50, 3))
print("Flagged appointments:", flagged_50)

Threshold: 0.50
Precision: 0.575
Recall: 0.249
F1-score: 0.347
Flagged appointments: 73


In [ ]:
pred_35 = (oof >= 0.35).astype(int)

precision_35 = precision_score(y_train, pred_35)
recall_35 = recall_score(y_train, pred_35)
f1_35 = f1_score(y_train, pred_35)

flagged_35 = pred_35.sum()

print("Threshold: 0.35")
print("Precision:", round(precision_35, 3))
print("Recall:", round(recall_35, 3))
print("F1-score:", round(f1_35, 3))
print("Flagged appointments:", flagged_35)

Threshold: 0.35
Precision: 0.513
Recall: 0.467
F1-score: 0.489
Flagged appointments: 154


A threshold of 0.35 was chosen because it gave a F1-score compared to the 0.50 threshold. Even though this lower threshold means more appointments are flagged for follow-up, it boosts recall. This helps the clinic catch potential no-shows, which is important for planning and reducing missed appointments.

In [ ]:
lr_pipe.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000))])

In [ ]:
probs = lr_pipe.predict_proba(X_test)[:, 1]

print("Total test probabilities:", len(probs))
print("First 10 probabilities:", probs[:10])

Total test probabilities: 198
First 10 probabilities: [0.02156542 0.15091659 0.51869182 0.19130033 0.16771795 0.31997718
 0.33844789 0.36703784 0.10499898 0.17326066]


In [ ]:
pred_test = (probs >= 0.35).astype(int)

print("Total predictions:", len(pred_test))
print("Flagged appointments:", pred_test.sum())

Total predictions: 198
Flagged appointments: 49


In [ ]:
precision_test = precision_score(y_test, pred_test)

recall_test = recall_score(y_test, pred_test)

print("Precision:", round(precision_test, 3))
print("Recall:", round(recall_test, 3))

Precision: 0.571
Recall: 0.571


In [ ]:
roc_auc_test = roc_auc_score(y_test, probs)

print("Test ROC-AUC:", round(roc_auc_test, 3))

Test ROC-AUC: 0.789


Our model flagged 49 out of 198 appointments for follow-up calls.

Approximately 57% of flagged appointments were actual no-shows, and the model identified approximately 57% of all actual no-shows.

The ROC-AUC of 0.79 indicates good ability to distinguish between the two appointment outcomes.

In [ ]:
print("X_test first 10 indices:")
print(X_test.index[:10].tolist())

print("\nX_test shape:")
print(X_test.shape)

X_test first 10 indices:
[213, 331, 501, 309, 88, 535, 280, 107, 59, 506]

X_test shape:
(198, 11)


In [ ]:
test_ids = df.loc[X_test.index, "appointment_id"]

print(test_ids.head(10))
print("\nTotal appointment IDs:", len(test_ids))

213    A0217
331    A0336
501    A0508
309    A0314
88     A0089
535    A0543
280    A0284
107    A0109
59     A0060
506    A0513
Name: appointment_id, dtype: object

Total appointment IDs: 198


In [ ]:
test_risk_scores = pd.DataFrame({
    "appointment_id": test_ids,
    "no_show": y_test,
    "risk_prob": probs,
    "flag": pred_test
})

print(test_risk_scores.head(10))
print("\nDataset shape:", test_risk_scores.shape)

    appointment_id  no_show  risk_prob  flag
213          A0217        0   0.021565     0
331          A0336        0   0.150917     0
501          A0508        0   0.518692     1
309          A0314        0   0.191300     0
88           A0089        0   0.167718     0
535          A0543        1   0.319977     0
280          A0284        0   0.338448     0
107          A0109        1   0.367038     1
59           A0060        0   0.104999     0
506          A0513        0   0.173261     0

Dataset shape: (198, 4)


In [ ]:
print("Total rows:", len(test_risk_scores))
print("Total columns:", len(test_risk_scores.columns))
print("Column names:", test_risk_scores.columns.tolist())
print("Flagged appointments:", test_risk_scores["flag"].sum())

# Check for missing values
print("\nMissing values:")
print(test_risk_scores.isnull().sum())

Total rows: 198
Total columns: 4
Column names: ['appointment_id', 'no_show', 'risk_prob', 'flag']
Flagged appointments: 49

Missing values:
appointment_id    0
no_show           0
risk_prob         0
flag              0
dtype: int64


In [ ]:
test_risk_scores.to_csv("test_risk_scores.csv", index=False)

# Task 4.1

In [ ]:
!pip install -q google-genai pydantic

In [ ]:
from google import genai
from google.genai import types
import json

API_KEY = input("Paste your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

print("Gemini client initialized successfully!")

Paste your Gemini API key: AQ.Ab8RN6IhxD6czY8hOtoMyMjxu6loHN3vjL1MAhanUXdettqs4Q
Gemini client initialized successfully!


In [ ]:
MODEL_NAME = "gemini-3.5-flash"
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

In [ ]:
SYSTEM_TEXT = """
You are a healthcare call-centre analysis assistant.
Analyse patient follow-up notes accurately.
Do not invent information or assume reasons that are not stated.
Return a response that follows the provided JSON schema.
"""


def build_prompt(note_text):
    """
    Builds a CRAFT-based prompt for analysing a patient follow-up note.

    Parameters:
        note_text (str): The patient's call-centre follow-up note.

    Returns:
        str: A formatted prompt containing context, role, action,
             tone, format instructions and the patient note.
    """

    prompt = f"""
    CONTEXT:
    A clinic wants to understand why patients miss appointments.

    ROLE:
    You are a healthcare call-centre analyst.

    ACTION:
    Read the patient follow-up note and identify the most likely
    reason for the missed appointment.

    Allowed root_cause values:
    - forgot_or_no_reminder
    - distance_or_transport
    - cost_or_fee
    - illness_or_emergency
    - work_or_schedule
    - other

    If the reason is unclear or does not fit the categories,
    select other. Do not guess.

    TONE:
    Be factual, neutral and concise.

    FORMAT:
    Return exactly these four fields:
    root_cause, confidence, summary, recommended_action.

    The confidence must be between 0 and 1.
    The summary MUST contain no more than 25 words.
    Count the words before responding.
    Do not exceed this limit under any circumstances.

    PATIENT NOTE:
    {note_text}
    """

    return prompt

In [ ]:
total_api_calls = 0

In [ ]:
LAST_RAW_RESPONSE = ""

def analyse_note(note_text):
    """
    Sends one patient follow-up note to Gemini for structured analysis.

    Parameters:
        note_text (str): The patient's call-centre follow-up note.

    Returns:
        dict: Parsed JSON containing exactly four fields:
            - root_cause
            - confidence
            - summary
            - recommended_action

    Process:
        1. Builds the CRAFT prompt.
        2. Sends it to the Gemini Flash model.
        3. Receives and preserves the original JSON response.
        4. Converts the JSON response into a Python dictionary.
    """

    global LAST_RAW_RESPONSE
    LAST_RAW_RESPONSE = ""

    global total_api_calls
    total_api_calls += 1

    prompt = build_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    LAST_RAW_RESPONSE = response.text

    data = json.loads(response.text)

    return data


In [ ]:
note_n001 = "Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week."

result_n001 = analyse_note(note_n001)

print(result_n001)

{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'Patient missed the appointment because she forgot about it after receiving no reminder message. She requested to reschedule.', 'recommended_action': "Reschedule the patient's appointment for next week and verify her contact details to ensure future SMS or email reminders are properly received."}


In [ ]:
summary_words = len(result_n001["summary"].split())

print("Summary word count:", summary_words)
print("Within 25-word limit:", summary_words <= 25)

Summary word count: 18
Within 25-word limit: True


# Task 4.2

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class FollowupInsight(BaseModel):

    root_cause: str = Field(min_length=3)

    confidence: float = Field(ge=0, le=1)

    summary: str = Field(min_length=5)

    recommended_action: str = Field(min_length=5)

In [ ]:
data = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 0.95,
    "summary": "Patient forgot appointment",
    "recommended_action": "Send appointment reminder"
}

In [ ]:
insight = FollowupInsight(**data)

print(insight)

root_cause='forgot_or_no_reminder' confidence=0.95 summary='Patient forgot appointment' recommended_action='Send appointment reminder'


In [ ]:
ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

In [ ]:
def validate_root_cause(root_cause):
    """
    Checks whether a root cause belongs to the approved categories.

    Parameters:
        root_cause (str): The root cause returned by Gemini.

    Returns:
        bool: True if the category is allowed, otherwise False.
    """

    return root_cause in ALLOWED_ROOT_CAUSES

print(validate_root_cause("forgot_or_no_reminder"))
print(validate_root_cause("personal_reason"))

True
False


In [ ]:
import pandas as pd

df_notes = pd.read_csv("/content/followup_notes.csv")

print(df_notes.head(10))
print(df_notes.columns)
print(df_notes.shape)

  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
3    N004          A0926                                                NaN
4    N005          A0534  Had high fever and body ache on the day, staye...
5    N006          A0507  Manager called an urgent meeting at the office...
6    N007          A0333  Bhool gaya, SMS nahi aaya tha. Dobara booking ...
7    N008          A0222  City bus strike that day, no auto available ne...
8    N009          A0977  Phone was switched off, could not reach the pa...
9    N010          A0596  Had a shift change at the factory and could no...
Index(['note_id', 'appointment_id', 'note_text'], dtype='object')
(30, 3)


In [ ]:
df_first10 = df_notes.head(10).copy()

print(df_first10.shape)

(10, 3)


In [ ]:
for _, row in df_first10.iterrows():

    note_text = row["note_text"]

    if pd.isna(note_text) or str(note_text).strip() == "":
        print(row["note_id"], "Blank note")
    else:
        print(row["note_id"], "Note available")

N001 Note available
N002 Note available
N003 Note available
N004 Blank note
N005 Note available
N006 Note available
N007 Note available
N008 Note available
N009 Note available
N010 Note available


In [ ]:
results = []

In [ ]:
for _, row in df_first10.iterrows():

    note_text = row["note_text"]

    if pd.isna(note_text) or str(note_text).strip() == "":

        results.append({
            "note_id": row["note_id"],
            "appointment_id": row["appointment_id"],
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        })

In [ ]:
print(results)
print("Total records stored:", len(results))

[{'note_id': 'N004', 'appointment_id': 'A0926', 'root_cause': 'other', 'confidence': 0.0, 'summary': 'No note recorded', 'recommended_action': 'Re-call patient', 'validation_status': 'skipped_blank', 'raw_response': ''}]
Total records stored: 1


In [ ]:
import time

results = []
api_calls = 0


for _, row in df_first10.iterrows():

    note_text = row["note_text"]


    data = {}
    raw_response = ""

    if pd.isna(note_text) or str(note_text).strip() == "":


        root_cause = "other"
        confidence = 0.0
        summary = "No note recorded"
        recommended_action = "Re-call patient"
        validation_status = "skipped_blank"

    else:


        if api_calls > 0:
            time.sleep(13)


        api_calls = api_calls + 1


        try:


            data = analyse_note(note_text)


            raw_response = LAST_RAW_RESPONSE


            insight = FollowupInsight(**data)


            if validate_root_cause(insight.root_cause):
                validation_status = "valid"
            else:
                validation_status = "invalid"


            root_cause = data.get("root_cause", "")
            confidence = data.get("confidence", "")
            summary = data.get("summary", "")
            recommended_action = data.get("recommended_action", "")


        except Exception as e:

            raw_response = LAST_RAW_RESPONSE

            validation_status = "invalid"


            root_cause = data.get("root_cause", "")
            confidence = data.get("confidence", "")
            summary = data.get("summary", "")
            recommended_action = data.get("recommended_action", "")

            print("Error in", row["note_id"], ":", e)


    result = {
        "note_id": row["note_id"],
        "appointment_id": row["appointment_id"],
        "root_cause": root_cause,
        "confidence": confidence,
        "summary": summary,
        "recommended_action": recommended_action,
        "validation_status": validation_status,
        "raw_response": raw_response
    }

    results.append(result)



print("Total records:", len(results))
print("Total API calls:", api_calls)


Total records: 10
Total API calls: 9


In [22]:
results

[{'note_id': 'N001',
  'appointment_id': 'A0872',
  'root_cause': 'forgot_or_no_reminder',
  'confidence': 0.95,
  'summary': 'Patient completely forgot about the visit because she never received a reminder message. She requested to rebook for next week.',
  'recommended_action': 'Reschedule the patient for next week and verify that her phone number and reminder preferences are correct in the system.',
  'validation_status': 'valid',
  'raw_response': '{"root_cause":"forgot_or_no_reminder","confidence":0.95,"summary":"Patient completely forgot about the visit because she never received a reminder message. She requested to rebook for next week.","recommended_action":"Reschedule the patient for next week and verify that her phone number and reminder preferences are correct in the system."}'},
 {'note_id': 'N002',
  'appointment_id': 'A0897',
  'root_cause': 'distance_or_transport',
  'confidence': 0.95,
  'summary': 'The patient missed the appointment due to living far from the clinic an

In [23]:
df_results = pd.DataFrame(results)

# print(df_results.head(10))
print(df_results.shape)

df_results.to_csv("notes_analysed.csv", index=False)

(10, 8)


In [24]:
print("Shape:", df_results.shape)
print("Columns:", df_results.columns.tolist())

print(df_results["validation_status"].value_counts())

Shape: (10, 8)
Columns: ['note_id', 'appointment_id', 'root_cause', 'confidence', 'summary', 'recommended_action', 'validation_status', 'raw_response']
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64


# Task 4.3

In [25]:
bad_data_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot appointment",
    "recommended_action": "Send appointment reminder"
}

try:
    FollowupInsight(**bad_data_1)

except ValidationError as e:
    print(e)

1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal


In [26]:
bad_data_2 = {
    "root_cause": "cost_or_fee",
    "confidence": 0.85,
    "summary": "Consultation fee too high"
}

try:
    FollowupInsight(**bad_data_2)

except ValidationError as e:
    print(e)

1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'cost_or_f...sultation fee too high'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [27]:
def compare_prompt(prompt):
    """
    Sends a prompt to Gemini using the same settings as Task 4.1.

    Parameters:
        prompt (str): The prompt being tested (V1 or V2).

    Returns:
        dict: Parsed JSON response from Gemini.

    """

    global total_api_calls
    total_api_calls += 1

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    return json.loads(response.text)

In [28]:
note_7 = df_notes.loc[
    df_notes["note_id"] == "N007", "note_text"
].iloc[0]

note_9 = df_notes.loc[
    df_notes["note_id"] == "N009", "note_text"
].iloc[0]

print("N007:", note_7)
print("N009:", note_9)

N007: Bhool gaya, SMS nahi aaya tha. Dobara booking karni hai.
N009: Phone was switched off, could not reach the patient after three attempts.


In [29]:
import time

note_text = note_7

prompt_v1 = "Classify this note: " + note_text
prompt_v2 = build_prompt(note_text)

v1_n007 = compare_prompt(prompt_v1)
time.sleep(13)

v2_n007 = compare_prompt(prompt_v2)
time.sleep(13)

note_text = note_9

prompt_v1 = "Classify this note: " + note_text
prompt_v2 = build_prompt(note_text)


v1_n009 = compare_prompt(prompt_v1)
time.sleep(13)

v2_n009 = compare_prompt(prompt_v2)


print("All four API calls completed.")
print("N007 V1:", v1_n007)
print("N007 V2:", v2_n007)
print("N009 V1:", v1_n009)
print("N009 V2:", v2_n009)

All four API calls completed.
N007 V1: {'root_cause': 'Patient forgot and did not receive an SMS reminder', 'confidence': 0.95, 'summary': 'The patient missed the appointment because they forgot about it and claimed they did not receive an SMS reminder. They requested to rebook.', 'recommended_action': "Reschedule the patient's appointment and verify/update their phone number in the system to ensure SMS reminders are successfully delivered."}
N007 V2: {'root_cause': 'forgot_or_no_reminder', 'confidence': 0.95, 'summary': 'The patient missed the appointment because they forgot and did not receive an SMS reminder. They want to rebook.', 'recommended_action': "Reschedule the patient's appointment and verify that their phone number is correctly updated in the SMS reminder system."}
N009 V1: {'root_cause': 'Patient unreachable - Phone switched off', 'confidence': 0.98, 'summary': 'The patient could not be reached after three attempts because their phone was switched off.', 'recommended_acti

The V2 system is chosen for deployment because the V2 system consistently follows the approved root‑cause categories. In the case of N007, the V2 system correctly classified the note as forgot_or_no_reminder, while the V1 system returned a category. For N009, the V2 system correctly classified the note as other, whereas the V1 system generated a new category. Therefore, the CRAFT‑based V2 prompt provides reliable and standardized outputs for downstream analysis.

In [30]:
print("Total Gemini API calls:", total_api_calls)

Total Gemini API calls: 14


# Task 4.4

In [32]:
import pandas as pd


df_results = pd.read_csv("/content/notes_analysed.csv")
df_appointments = pd.read_csv("/content/appointments_clean.csv")

df_merged = pd.merge(
    df_results,
    df_appointments,
    on="appointment_id",
    how="left",
    validate="many_to_one"
)

display(df_merged)

print("Merged table shape:", df_merged.shape)
print("Total rows:", len(df_merged))
print(
    "Missing appointment matches:",
    df_merged["reminder_sent"].isna().sum()
)


assert len(df_merged) == 10, "Acceptance criteria failed: expected 10 rows."

print("Task 4.4(a): 10-row acceptance criterion passed.")

,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response,patient_id,clinic_id,...,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,N001,A0872,forgot_or_no_reminder,0.95,Patient completely forgot about the visit beca...,Reschedule the patient for next week and verif...,valid,"{""root_cause"":""forgot_or_no_reminder"",""confide...",P144,C05,...,F,17.6,750,App,Consultation,No,1,No-Show,14,1
1,N002,A0897,distance_or_transport,0.95,The patient missed the appointment due to livi...,Offer telehealth options or assist with schedu...,valid,"{""root_cause"":""distance_or_transport"",""confide...",P268,C07,...,M,17.2,1219,Phone,Follow-up,No,0,No-Show,5,1
2,N003,A0526,cost_or_fee,0.95,Patient skipped the appointment because the co...,Provide information on available financial ass...,valid,"{""root_cause"":""cost_or_fee"",""confidence"":0.95,...",P006,C04,...,F,5.3,800,App,Consultation,Yes,0,No-Show,34,1
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,NaN,P449,C04,...,M,4.5,300,Phone,Follow-up,No,0,No-Show,40,1
4,N005,A0534,illness_or_emergency,0.95,The patient missed the appointment due to a hi...,Follow up with the patient in a few days to re...,valid,"{""root_cause"":""illness_or_emergency"",""confiden...",P400,C04,...,F,4.2,550,App,Follow-up,Yes,0,No-Show,26,1
5,N006,A0507,work_or_schedule,0.95,Patient missed the appointment due to an urgen...,Reschedule the patient for an evening appointm...,valid,"{""root_cause"":""work_or_schedule"",""confidence"":...",P230,C04,...,F,0.8,700,App,Consultation,Yes,2,No-Show,28,1
6,N007,A0333,forgot_or_no_reminder,0.95,Patient forgot the appointment because no SMS ...,Reschedule the appointment and verify the pati...,valid,"{\n ""root_cause"": ""forgot_or_no_reminder"",\n ...",P472,C08,...,F,15.2,850,Walk-In,Consultation,No,2,No-Show,9,1
7,N008,A0222,distance_or_transport,0.95,Patient missed the appointment due to a city b...,Offer telehealth options or assist with altern...,valid,"{\n ""root_cause"": ""distance_or_transport"",\n ...",P125,C02,...,F,15.8,650,App,Follow-up,Yes,1,No-Show,4,1
8,N009,A0977,other,0.90,The patient could not be reached after three p...,Send an automated text message or email to pro...,valid,"{""root_cause"":""other"",""confidence"":0.9,""summar...",P334,C01,...,M,3.6,800,App,Consultation,Yes,1,No-Show,4,1
9,N010,A0596,work_or_schedule,0.95,Patient missed the appointment due to an unexp...,Reschedule the appointment for an available Sa...,valid,"{\n ""root_cause"": ""work_or_schedule"",\n ""con...",P297,C04,...,F,10.1,400,App,Follow-up,No,1,No-Show,21,1


Merged table shape: (10, 23)
Total rows: 10
Missing appointment matches: 0
Task 4.4(a): 10-row acceptance criterion passed.


In [33]:
root_cause_reminder_table = pd.crosstab(
    df_merged["root_cause"],
    df_merged["reminder_sent"],
    margins=True
)

print("Root Cause vs Reminder Sent")
display(root_cause_reminder_table)

Root Cause vs Reminder Sent


reminder_sent,No,Yes,All
root_cause,,,
cost_or_fee,0,1,1
distance_or_transport,1,1,2
forgot_or_no_reminder,2,0,2
illness_or_emergency,0,1,1
other,1,1,2
work_or_schedule,1,1,2
All,5,5,10


In [34]:
df_merged.to_csv("task_4_4_merged.csv", index=False)

root_cause_reminder_table.to_csv("task_4_4_crosstab.csv")

print("Saved task_4_4_merged.csv")
print("Saved task_4_4_crosstab.csv")

Saved task_4_4_merged.csv
Saved task_4_4_crosstab.csv


In [35]:
print("Root cause counts, including missing values:")
print(df_merged["root_cause"].value_counts(dropna=False))

print("\nValidation status:")
print(df_merged["validation_status"].value_counts(dropna=False))

print("\nRoot cause and reminder status:")
display(
    df_merged[
        ["note_id", "root_cause", "validation_status", "reminder_sent"]
    ]
)

Root cause counts, including missing values:
root_cause
forgot_or_no_reminder    2
distance_or_transport    2
other                    2
work_or_schedule         2
cost_or_fee              1
illness_or_emergency     1
Name: count, dtype: int64

Validation status:
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64

Root cause and reminder status:


,note_id,root_cause,validation_status,reminder_sent
0,N001,forgot_or_no_reminder,valid,No
1,N002,distance_or_transport,valid,No
2,N003,cost_or_fee,valid,Yes
3,N004,other,skipped_blank,No
4,N005,illness_or_emergency,valid,Yes
5,N006,work_or_schedule,valid,Yes
6,N007,forgot_or_no_reminder,valid,No
7,N008,distance_or_transport,valid,Yes
8,N009,other,valid,Yes
9,N010,work_or_schedule,valid,No
